# 프롬프트 추론 실험 조종석 (exp07 고정)

**목적**: 가중치 업데이트 없이 프롬프트 변형·힌트 주입·취약 세그먼트 라우팅으로
exp07(shuffled 48.4%)의 약점을 보완할 수 있는지 사전 스크리닝.
상세 계획: `PLAN_prompt_experiments.md` | 검토 배경: `PLAN_prompt_and_preprocessing.md`

**라운드 순서 (7/15 확정)**
R0 검증 → **R1** 기본 프롬프트 소변형 → **R2** v4(팀 제안) 기능 분해 → **R3** 주입형 힌트(CLIP 등)
→ **R4** 취약 세그먼트 라우팅(앞 라운드 승자 투입) → **R5** CoT 소액 베팅

**규칙**
- GPU 하나 = 학습(Train_Experiments.ipynb)과 **동시 실행 금지**. 모델 로드 셀이 VRAM 확보까지 대기함
- exp07은 v1_list로 학습됨 → 프롬프트 대변경은 하락이 기본 기대값. 판정은 **상대 비교 + paired(fixed/broken)**
- greedy(do_sample=False)라 결과는 결정론적 — baseline 재추론 불필요 (기존 예측 파일 재사용)
- 기록: `outputs/prompt_experiments.csv` | 판정: shuffled ±4%p(≈10문제) 노이즈, 취약 세그먼트는 fixed−broken 차이로

In [ ]:
# ① 준비 — holdout 300 + 문장유형/ai_score/CLIP 피처 + exp07 baseline (모델 로드 없음, ~30초)
import sys; sys.path.insert(0, "scripts")
import importlib
import prompt_lab as lab; importlib.reload(lab)
import prompts as prompt_registry; importlib.reload(prompt_registry)

df = lab.load_holdout()          # Partition/ai_score/N플래그/CLIP 거리/weak 마스크 포함
baseline = lab.load_baseline(df) # exp07 v1_list 기존 예측 (재현 동일하므로 재추론 불필요)
print("baseline shuffled:", round(baseline[~baseline.No_ordering].correct.mean(), 4), "(기대 0.4841)")

In [ ]:
# ② 모델 로드 (1회, ~2분) — 이후 모든 변형이 재사용
model, processor = lab.load_model()

In [ ]:
# ③ 변형 레지스트리 — 실험 추가 = 여기 한 줄. text_fn(row) -> 유저 텍스트
V1 = prompt_registry.PROMPTS["v1_list"]
def v1(row): return V1.format(sentence=row.Sentence)

# --- R1: 기본 프롬프트 소변형 재료
CAUSAL = (" If the sentence has few explicit time cues, infer the order from cause and"
          " effect (for example: clean to dirty, empty to full, intact to broken).")
VISUAL = (" Pay close attention to visual state changes across the images, such as"
          " objects appearing, moving, or changing appearance.")
REORDER = ('Look at the 4 images above labeled Image 1 to Image 4. Determine the correct '
           'chronological order of these images to match the sentence below.\n'
           'Sentence: "{sentence}"\n'
           "Provide the answer ONLY as a Python list of integers. Example: [1, 2, 3, 4]")
# --- R2: v4(팀 제안 7/15) 기능 분해 재료 (r2_combo 본체는 prompts.py의 v4_story)
ROLE = ("You are an expert visual storyteller and video editor. "
        "Your task is to reconstruct the correct chronological order of "
        "4 shuffled video frames based on the sentence.\n")
STORY = ("Your task is to reconstruct the correct chronological order of 4 shuffled "
         "video frames based on a provided storyline.\n"
         'Storyline: "{sentence}"\n'
         "Look at the 4 images above labeled Image 1 to Image 4. "
         "Determine the correct chronological order of these images to match the storyline. "
         "Provide the answer ONLY as a Python list of integers. Example: [1, 2, 3, 4]")
CUES = (" Identify key visual cues in each image (object states, character actions, "
        "background changes) that match the events in the sentence.")
# --- R3: 주입형 힌트 재료
def clip_hint(row):
    return (f"\nHint: the most visually similar image pairs are {lab.clip_pairs_text(row)}. "
            "Chronologically adjacent frames are often visually similar.")
# --- R5: CoT 재료 (경량판 — 풀버전은 prompts.py의 v4_story_cot)
COT = ('Sentence: "{sentence}"\n'
       "Follow these steps strictly in this order:\n"
       "Step 1. Note the temporal cues in the sentence (words like begins, then, after, finally).\n"
       "Step 2. Briefly compare the 4 images above (Image 1 to Image 4) and match each to a part of the sentence.\n"
       "Step 3. Combine the cues and comparisons to determine the chronological order.\n"
       "End your answer with ONLY a Python list of integers on the last line. Example: [1, 2, 3, 4]")

# name -> (text_fn, kwargs)  |  subset="weak"이면 취약 세그먼트만 추론(나머지 baseline 유지)
VARIANTS = {
    # R0 — 파이프라인 검증: 기존 48.41%가 재현되어야 정상
    "r0_v1_baseline": (v1, {}),
    # R1 — 기본 프롬프트(v1) 소변형 민감도 (각 ~5분)
    "r1_causal":  (lambda row: v1(row) + CAUSAL, {}),
    "r1_visual":  (lambda row: v1(row) + VISUAL, {}),
    "r1_reorder": (lambda row: REORDER.format(sentence=row.Sentence), {}),  # 구조 변형 대조군
    # R2 — v4(팀 제안) 기능 분해: 역할/스토리 프레이밍/단서 가이드 단독 + 결합 (각 ~5.5분)
    "r2_role":  (lambda row: ROLE + v1(row), {}),
    "r2_story": (lambda row: STORY.format(sentence=row.Sentence), {}),
    "r2_cues":  (lambda row: v1(row) + CUES, {}),
    "r2_combo": (lambda row: prompt_registry.build_user_text("v4_story", row.Sentence), {}),
    # R3 — 현재 exp07에 적용 가능한 주입형 힌트 (전체 300, exp09 등록 후보 선별)
    "r3_clip_pairs": (lambda row: v1(row) + clip_hint(row), {}),
    # R4 — 취약 세그먼트 라우팅 (각 ~1분, Type-1 or ai_score>0.5만 재추론)
    #      ↓ 기본 2개 + R1~R3 승자가 나오면 같은 패턴으로 한 줄 추가
    "r4_route_causal": (lambda row: v1(row) + CAUSAL, {"subset": "weak"}),
    "r4_route_clip":   (lambda row: v1(row) + clip_hint(row), {"subset": "weak"}),
    # R5 — CoT 소액 베팅 (취약 41개만): 경량 CoT + 팀 제안 풀버전 원문
    "r5_cot256_weak":  (lambda row: COT.format(sentence=row.Sentence),
                        {"max_new_tokens": 256, "subset": "weak"}),
    "r5_fullcot_weak": (lambda row: prompt_registry.build_user_text("v4_story_cot", row.Sentence),
                        {"max_new_tokens": 512, "subset": "weak"}),
}

sample = df[df.weak].iloc[0]  # 프롬프트 미리보기 (취약 샘플 기준)
for n, (fn, kw) in VARIANTS.items():
    print(f"=== {n} {kw} ===\n{fn(sample)}\n")

## R0 — 파이프라인 검증 (~5분)
`r0_v1_baseline`이 기존 48.41%와 **정확히 일치**해야 이후 모든 비교가 유효하다.
(greedy 재현 확인 — 다르면 max_pixels/어댑터 경로부터 의심)

In [ ]:
def run(name):
    fn, kw = VARIANTS[name]
    kw = dict(kw)
    if kw.get("subset") == "weak": kw["subset"] = df["weak"]
    return lab.run_variant(name, fn, model, processor, df, baseline=baseline, **kw)

res0 = run("r0_v1_baseline")
assert abs(res0[~res0.No_ordering].correct.mean() - 0.4841) < 1e-3, "baseline 재현 실패!"
print("재현 OK")

## R1 — 기본 프롬프트 소변형 민감도 (~15분)
목적은 개선이 아니라 "얼마나 바꾸면 얼마나 깨지는지" 곡선 확보.
r1_reorder(구조 변형)가 크게 무너지면 이후 변형은 문구 추가 수준으로 제한.

In [ ]:
for name in ["r1_causal", "r1_visual", "r1_reorder"]:
    run(name)
lab.show_results()

## R2 — v4(팀 제안) 기능 분해 (~22분)
역할 부여 / 스토리라인 프레이밍 / 시각 단서 가이드를 **하나씩 분리**해 어떤 요소가
효과·해가 되는지 측정. `r2_combo` = prompts.py `v4_story` = 팀 제안의 축약 직답판 전체
(→ 미니 학습 후보이기도 함). 여기서 이기는 요소가 R3 주입·R4 라우팅의 재료가 된다.

In [ ]:
for name in ["r2_role", "r2_story", "r2_cues", "r2_combo"]:
    run(name)
lab.show_results()

## R3 — 현재 exp07에 적용 가능한 주입형 힌트 (~5분+)
학습 없이 주입이므로 하락 예상 — **절대치가 아니라 형식 간 상대 비교(acc_weak 델타)**로
exp09(힌트 SFT)에 등록할 문구 후보를 고른다.
R2 승자 요소가 있으면 `v1(row) + 승자요소 + clip_hint(row)` 조합 변형을 여기 추가.

In [ ]:
for name in ["r3_clip_pairs"]:
    run(name)
lab.show_results()

## R4 — 취약 세그먼트 라우팅 (핵심, 변형당 ~1분)
Type-1 또는 ai_score>0.5(현재 acc ~22%)만 재추론, 나머지는 baseline 유지
→ 하락 위험이 취약 세그먼트로 국한. **채택 기준: fixed−broken ≥ +10 (전체 +4%p)**
R1~R3 승자를 `{"subset": "weak"}`로 레지스트리에 추가해 여기서 재실행.

In [ ]:
for name in ["r4_route_causal", "r4_route_clip"]:
    run(name)
lab.show_results()

## R5 — CoT 소액 베팅 (취약 41개만, ~7분)
- 스모크에서 exp07이 단계 지시를 무시하고 리스트 즉답하는 것 확인됨 → 기대치 낮음, 기록 목적
- `r5_cot256_weak`: 경량 단계 유도 | `r5_fullcot_weak`: 팀 제안 풀버전 원문(<ANSWER> 태그, 512토큰)
- 여기서 양수 신호(fixed>broken)면 CoT 학습 실험(다음 단계) 우선순위 상향 근거
- raw 출력 질적 확인: `outputs/preds/prompt_r5_*.csv`의 raw 컬럼

In [ ]:
for name in ["r5_cot256_weak", "r5_fullcot_weak"]:
    run(name)
lab.show_results()

## 종합 판정
- `acc_shuffled` + `fixed/broken`(paired)으로 정렬 비교
- 세그먼트 분해로 풍선 효과(Type-2 하락) 감시

In [ ]:
import pandas as pd
display(lab.show_results())
# 관심 변형의 세그먼트 분해 (이름 바꿔가며)
res = pd.read_csv("./outputs/preds/prompt_r2_combo.csv")
display(lab.segment_table(res))
# baseline 대비 뭘 새로 맞추고 틀렸는지 샘플 확인
b = baseline.set_index("Id").correct
r = res.set_index("Id").correct
print("[새로 맞춘 샘플]"); display(res[res.Id.isin(r[r & ~b].index)][["Id","Partition","ai_score","Sentence"]].head(10))
print("[새로 틀린 샘플]"); display(res[res.Id.isin(r[~r & b].index)][["Id","Partition","ai_score","Sentence"]].head(10))

## 제출 파일 생성 (실험 확정 후)
확정 변형으로 test 819개 추론 → `outputs/submissions/submission_<이름>_<시각>.csv`
- 소요 ~13분 (CoT 변형이면 ~2h; 규정 한도 24h). **제출은 팀 전체 1일 2회** — holdout 검증 끝난 변형만
- 형식 검증(행 수·Id 집합·순열 유효성) 자동 — 실패 시 파일을 만들지 않고 assert로 멈춤
- 라우팅 변형(subset="weak")을 넣으면 test에서도 같은 라우터(Type-1 or ai_score>0.5)가 적용됨
- CLIP 힌트 변형은 test용 CLIP 피처(`snu_clip_features_test.csv`) 생성 후 사용 가능 (전처리 트랙 TODO)

In [ ]:
import importlib, prompt_lab as lab
importlib.reload(lab)              # 수정된 코드 강제 반영

test_df = lab.load_testset()
for FINAL in ["r0_v1_baseline", "r2_combo"]:
    fn, kw = VARIANTS[FINAL]
    lab.make_submission(FINAL, fn, model, processor, test_df,
                        max_new_tokens=kw.get("max_new_tokens", 32))

: 